## Outline

- DF for minute features at date grain (Done)
- DF for daily features at date grain (Done)
- DF for returns over 1, 3 and 5 days (Done)
- Simple logistic, rfc and xgb models for daily alone, min alone and then combined
- Permutation importance
- Chart over rolling 5 days for 25 iterations, aka 6 months

In [34]:
import min_features, daily_return
import importlib
import pandas as pd

importlib.reload(min_features)
importlib.reload(daily_return)

df_min = min_features.min_features()
returns = [1, 3, 5]
df_daily = daily_return.pull_daily('QQQ', returns) 

df_main = pd.merge(df_min, df_daily, how='inner', on='Date')
df_main = df_main.sort_values(by='Date', ascending=False)

In [61]:
return_cols = df_main.columns[df_main.columns.str.contains("Return_")].to_list()
daily_cols = df_daily.iloc[:, 1:].columns.difference(return_cols).to_list()
min_cols = df_min.iloc[:, 1:].columns.to_list()

In [63]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import TimeSeriesSplit
import numpy as np


column_sets = [daily_cols]#, min_cols, daily_cols + min_cols]
names = ['daily']#, 'minute', 'daily+minute']
returns = [1]#, 3, 5]
runs = 5
test_size = 5
lbs = [6]
offset_size = test_size

models = {
    "logistic": LogisticRegression(max_iter=1000),
    "linear_svm": LinearSVC(),
    "random_forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1,
    ),
    "grad_boost": GradientBoostingClassifier(random_state=42),
    "naive_bayes": GaussianNB(),
}

tscv = TimeSeriesSplit(n_splits=5)

results = []

for feature_cols, name in zip(column_sets, names):

    X = df_main[feature_cols].to_numpy()

    for r in returns:
        
        y = df_main[f"Return_{r}"].to_numpy()
        
        for i in range(runs): # number of runs to do

            offset = max(i * offset_size, 0) # step size for each run
            
            for lb in lbs:
                 
                df_ph = df_main.iloc[offset : offset + 245 * lb, :].copy()  # 245 records is ~1 year of data
                ret_col = f"Return_{r}"
                ret_pct_col = f"Return%_{r}"

                rets = df_ph[ret_pct_col]
                neg, pos = rets[rets < 0], rets[rets > 0]

                neg_cut = neg.nlargest(max(1, int(len(neg) * 0.05))).min()
                pos_cut = pos.nsmallest(max(1, int(len(pos) * 0.05))).max()
                filtered = df_ph[(rets < neg_cut) | (rets > pos_cut)].copy()
                print(f"Run {i+1} of {runs} | LB: {lb} | Horizon: {r} | {filtered['Date'].iloc[test_size]} - {filtered['Date'].iloc[0]}")

                df_indicators = filtered[feature_cols]
                df_indicators = df_indicators.replace([np.inf, -np.inf], 0)
                df_predict = filtered[ret_col]


Run 1 of 5 | LB: 6 | Horizon: 1 | 2025-12-12 - 2025-12-19
Run 2 of 5 | LB: 6 | Horizon: 1 | 2025-12-04 - 2025-12-12
Run 3 of 5 | LB: 6 | Horizon: 1 | 2025-11-25 - 2025-12-05
Run 4 of 5 | LB: 6 | Horizon: 1 | 2025-11-19 - 2025-11-26
Run 5 of 5 | LB: 6 | Horizon: 1 | 2025-11-10 - 2025-11-19
